In [1]:
# 📦 Bibliotheken importieren
import os
import json
import yfinance as yf
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense
from datetime import datetime
import tensorflow as tf

if not tf.executing_eagerly():
    tf.compat.v1.enable_eager_execution()

# 🧩 Hyperparameter & Einstellungen
a = 75
b = 300

forecast_days = 5
anfang = "1990-01-01"
ende = datetime.today().strftime("%Y-%m-%d")
anzeige_tage = 200
days2learn2predict = anzeige_tage

os.makedirs("trainingsdaten", exist_ok=True)
def get_paths(ticker):
    base_dir = f"trainingsdaten/{ticker}"
    os.makedirs(base_dir, exist_ok=True)
    return {
        "model": f"{base_dir}/{ticker}_lstm_model.keras",
        "scaler": f"{base_dir}/{ticker}_scaler.save",
        "meta": f"{base_dir}/{ticker}_meta.json",
        "log": f"{base_dir}/training_log.txt"
    }

def prepare_data(data):
    close_prices = data[["Close"]].copy()
    scaler = MinMaxScaler()
    close_scaled = scaler.fit_transform(close_prices)
    X, y = [], []
    for i in range(days2learn2predict, len(close_scaled)):
        X.append(close_scaled[i - days2learn2predict:i, 0])
        y.append(close_scaled[i, 0])
    X, y = np.array(X), np.array(y)
    X = X.reshape((X.shape[0], X.shape[1], 1))
    return scaler, close_scaled, X, y

def create_model(units, input_shape):
    model = Sequential()
    model.add(LSTM(units=units, return_sequences=False, input_shape=input_shape))
    model.add(Dense(units=1))
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model

def train_and_save_model(model, X, y, model_path, scaler, scaler_path, meta_path, a, b, date):
    model.fit(X, y, epochs=b, batch_size=32)
    model.save(model_path)
    joblib.dump(scaler, scaler_path)
    with open(meta_path, "w") as f:
        json.dump({"a": a, "b": b, "last_date": date}, f)
    print("📦 Modell, Scaler & Meta-Daten gespeichert.")

def log_training(log_path, log_entry, ticker, a, b, date):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"[{timestamp}] {log_entry} | {ticker} | a={a} | b={b} | datenstand={date}\n"
    with open(log_path, "a") as log_file:
        log_file.write(log_line)

# ▶️ Hauptprogramm
def main():
    ticker = input("Bitte gib das Tickersymbol der Aktie ein (z. B. AAPL oder ENR.DE): ").upper()
    neu_trainieren = input(f"Möchtest du das Modell für {ticker} neu trainieren? (ja/nein): ").strip().lower() == "ja"

    paths = get_paths(ticker)
    log_path = paths["log"]
    data = yf.download(ticker, start=anfang, end=ende)
    print(data.head())

    plt.figure(figsize=(12, 6))
    plt.plot(data["Close"], label="Schlusskurs")
    plt.title(f"{ticker} Closing Prices")
    plt.xlabel("Datum")
    plt.ylabel("Preis")
    plt.grid(True)
    plt.legend()
    plt.show()

    scaler, close_scaled, X, y = prepare_data(data)
    print(f"Input-Shape: {X.shape}, Output-Shape: {y.shape}")

    model_exists = os.path.exists(paths["model"]) and os.path.exists(paths["scaler"]) and os.path.exists(paths["meta"])
    model_trained = False
    log_entry = None

    last_data_date = str(data.index[-1].date())
    saved_date = None
    if os.path.exists(paths["meta"]):
        with open(paths["meta"], "r") as f:
            meta = json.load(f)
            saved_date = meta.get("last_date")

    if neu_trainieren:
        print("🔁 Training wurde manuell gestartet.")
        model = create_model(a, (X.shape[1], 1))
        train_and_save_model(model, X, y, paths["model"], scaler, paths["scaler"], paths["meta"], a, b, last_data_date)
        log_entry = "manuell_neu"
        model_trained = True
    elif model_exists and saved_date == last_data_date:
        print("✅ Modell ist aktuell – lade gespeichertes Modell.")
        model = load_model(paths["model"])
        scaler = joblib.load(paths["scaler"])
        log_entry = "geladen"
    elif model_exists:
        print("📅 Neue Daten erkannt – weitertrainieren ...")
        model = load_model(paths["model"])
        scaler = joblib.load(paths["scaler"])
        model.fit(X, y, epochs=b, batch_size=32)
        model.save(paths["model"])
        with open(paths["meta"], "w") as f:
            json.dump({"last_date": last_data_date}, f)
        log_entry = "weiter"
        model_trained = True
    else:
        print("🚀 Kein Modell vorhanden – trainiere neu ...")
        model = create_model(a, (X.shape[1], 1))
        train_and_save_model(model, X, y, paths["model"], scaler, paths["scaler"], paths["meta"], a, b, last_data_date)
        log_entry = "neu"
        model_trained = True

    if log_entry:
        log_training(log_path, log_entry, ticker, a, b, last_data_date)

    predicted_scaled = model.predict(X)
    predicted_prices = scaler.inverse_transform(predicted_scaled)
    real_prices = scaler.inverse_transform(y.reshape(-1, 1))
    dates = data.index[days2learn2predict:]
    display_start = max(0, len(dates) - anzeige_tage)

    plt.figure(figsize=(12, 6))
    plt.plot(dates[display_start:], real_prices[display_start:], label="Echte Kurse")
    plt.plot(dates[display_start:], predicted_prices[display_start:], label="Vorhergesagte Kurse")
    plt.title(f"Echte vs. Vorhergesagte Kurse für {ticker}")
    plt.xlabel("Datum")
    plt.ylabel("Preis")
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    last_prediction_days = close_scaled[-days2learn2predict:].reshape(1, days2learn2predict, 1)
    predictions = []
    for _ in range(forecast_days):
        pred = model.predict(last_prediction_days)[0, 0]
        predictions.append(pred)
        last_prediction_days = np.append(last_prediction_days[:, 1:, :], [[[pred]]], axis=1)

    predictions_prices = scaler.inverse_transform(np.array(predictions).reshape(-1, 1))
    last_date = data.index[-1]
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=forecast_days)

    historical_display_dates = data.index[-forecast_days:]
    historical_display_prices = data["Close"][-forecast_days:]

    plt.figure(figsize=(12, 6))
    plt.plot(historical_display_dates, historical_display_prices, label="Echte Kurse")
    plt.plot(future_dates, predictions_prices, marker='o', linestyle='--', color='red', label=f"{forecast_days}-Tage Vorhersage")
    plt.title(f"{ticker} Kurs & Vorhersage ({forecast_days} Tage)")
    plt.xlabel("Datum")
    plt.ylabel("Preis")
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    delta = data["Close"].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean().replace(0, np.nan)
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    data["RSI"] = rsi

    diffs = np.diff(predictions_prices.flatten())
    average_slope = np.mean(diffs)
    print(f"📐 Durchschnittliche tägliche Steigung der Vorhersagekurve: {average_slope:.4f} pro Tag")

    recent_rsi = data["RSI"].dropna().iloc[-forecast_days:]
    if not recent_rsi.empty:
        average_rsi = recent_rsi.mean()
        print(f"📊 RSI der letzten {forecast_days} Tage: {average_rsi:.2f} (<30% kaufen, >70% verkaufen)")
    else:
        print(f"⚠️ Nicht genug Daten, um RSI-Durchschnitt für {forecast_days} Tage zu berechnen.")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'yfinance'